In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor, ASTForAudioClassification
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import wandb

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

2026-03-28 18:39:10.442666: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774723150.647579      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774723150.707919      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774723151.192215      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774723151.192260      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774723151.192263      24 computation_placer.cc:177] computation placer alr

In [3]:
DATA_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
BASE_PATH = DATA_PATH   

SAMPLE_RATE = 16000
CROP_DURATION = 30
SNR_MIN = 5
SNR_MAX = 20

STEMS = ["drums", "vocals", "bass", "other"]

GENRES = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

GENRE_TO_IDX = {genre: idx for idx, genre in enumerate(GENRES)}
IDX_TO_GENRE = {idx: genre for genre, idx in GENRE_TO_IDX.items()}

# OPTIMIZED TRAINING CONFIG
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 3e-5
NUM_WORKERS = 4

# OPTIMIZED DATA CONFIG
SAMPLES_PER_GENRE_TRAIN = 150
SAMPLES_PER_GENRE_VAL = 30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
def load_audio(filepath, sr=SAMPLE_RATE):
    """Load audio file"""
    try:
        audio, _ = librosa.load(filepath, sr=sr, mono=True)
        return audio
    except Exception as e:
        print(f"Error loading: {filepath}")
        return None

def crop_audio(audio, duration_sec, sr=SAMPLE_RATE):
    """Crop or pad audio to fixed duration"""
    target_length = duration_sec * sr
   
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        start = random.randint(0, len(audio) - target_length)
        audio = audio[start:start + target_length]
   
    return audio
    
def add_noise(audio, noise, snr_db):
    """Add noise to audio at specific SNR"""
    if len(noise) < len(audio):
        noise = np.tile(noise, int(np.ceil(len(audio)/len(noise))))
    noise = noise[:len(audio)]
   
    # Random position for noise
    start = random.randint(0, len(audio)//2)
    end = min(len(audio), start + len(noise)//2)
   
    segment = audio[start:end]
    noise_segment = noise[:end-start]
   
    signal_power = np.mean(segment ** 2)
    noise_power = np.mean(noise_segment ** 2)
   
    if noise_power > 0:
        snr_linear = 10 ** (snr_db / 10)
        scale = np.sqrt(signal_power / (noise_power * snr_linear))
        audio[start:end] += noise_segment * scale
   
    # Normalize
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    return audio

def get_noise_paths(data_path):
    """Get all noise file paths"""
    noise_path = Path(data_path) / "ESC-50-master" / "audio"
    if not noise_path.exists():
        return []
    return [str(f) for f in noise_path.glob("*.wav")]



In [5]:
class OptimizedMashupDataset(Dataset):
    """OPTIMIZED: Faster mashup generation - Tempo sync removed"""
   
    def __init__(self, data_path, genres, stems, noise_paths,
                 num_samples_per_genre=150, sr=SAMPLE_RATE,
                 crop_duration=CROP_DURATION):
       
        self.data_path = Path(data_path)
        self.genres = genres
        self.stems = stems
        self.noise_paths = noise_paths
        self.num_samples_per_genre = num_samples_per_genre
        self.sr = sr
        self.crop_duration = crop_duration
       
        self.stem_dict = self._build_stem_dict()
        self.total_samples = len(genres) * num_samples_per_genre
        
    def _build_stem_dict(self):
        stem_dict = {}
        for genre in self.genres:
            genre_path = self.data_path / "genres_stems" / genre
            stem_dict[genre] = {stem: [] for stem in self.stems}
           
            if not genre_path.exists():
                continue
               
            for song in os.listdir(genre_path):
                song_path = genre_path / song
                if not song_path.is_dir():
                    continue
               
                for stem in self.stems:
                    file = song_path / f"{stem}.wav"
                    if file.exists():
                        stem_dict[genre][stem].append(str(file))
       
        return stem_dict

    def _generate_mashup(self, genre):
        """OPTIMIZED: Faster mashup generation - NO TEMPO SYNC"""
       
        # Select random stems from different songs
        selected_paths = []
        for stem in self.stems:
            if self.stem_dict[genre][stem]:
                selected_paths.append(random.choice(self.stem_dict[genre][stem]))
       
        if len(selected_paths) < 2:
            return None
       
        # Load audio
        audios = []
        for path in selected_paths:
            y = load_audio(path, self.sr)
            if y is not None:
                audios.append(y)
       
        if len(audios) < 2:
            return None
       
        synced = audios   # Directly use loaded audio without tempo synchronization
        
        # Mix with random weights
        min_len = min(len(y) for y in synced)
        mix = np.zeros(min_len)
        
        for y in synced:
            weight = random.uniform(0.6, 1.4)
            mix += weight * y[:min_len]
       
        mix = mix / (np.max(np.abs(mix)) + 1e-8)
       
        # Crop
        mix = crop_audio(mix, self.crop_duration, self.sr)
       
        # Add noise (80% chance)
        if self.noise_paths and random.random() > 0.2:
            noise = load_audio(random.choice(self.noise_paths), self.sr)
            if noise is not None:
                noise = crop_audio(noise, self.crop_duration, self.sr)
                snr = random.uniform(SNR_MIN, SNR_MAX)
                mix = add_noise(mix, noise, snr)
       
        return mix

    def __len__(self):
        return self.total_samples
   
    def __getitem__(self, idx):
        """Generate mashup and return audio + label"""
       
        genre_idx = idx // self.num_samples_per_genre
        genre = self.genres[genre_idx]
       
        audio = None
        max_attempts = 3
        for _ in range(max_attempts):
            audio = self._generate_mashup(genre)
            if audio is not None:
                break
       
        if audio is None:
            audio = np.zeros(self.crop_duration * self.sr)
       
        label = GENRE_TO_IDX[genre]
       
        return audio, label

In [6]:
class ASTDataset(Dataset):
    """Dataset wrapper for AST model"""
   
    def __init__(self, base_dataset, feature_extractor):
        self.base_dataset = base_dataset
        self.feature_extractor = feature_extractor
   
    def __len__(self):
        return len(self.base_dataset)
   
    def __getitem__(self, idx):
        audio, label = self.base_dataset[idx]
       
        inputs = self.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        )
       
        return inputs.input_values.squeeze(0), label

In [7]:

def train_ast_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
   
    pbar = tqdm(dataloader, desc="Training AST")
   
    for inputs, labels in pbar:
        inputs = inputs.to(device)
        labels = labels.to(device)
       
        optimizer.zero_grad()
       
        outputs = model(inputs, labels=labels)
        loss = outputs.loss
       
        loss.backward()
        optimizer.step()
        scheduler.step()
       
        running_loss += loss.item()
       
        logits = outputs.logits
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
       
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
   
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
   
    return epoch_loss, epoch_acc


def validate_ast(model, dataloader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
   
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Validating AST"):
            inputs = inputs.to(device)
            labels = labels.to(device)
           
            outputs = model(inputs, labels=labels)
            loss = outputs.loss
           
            running_loss += loss.item()
           
            logits = outputs.logits
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
   
    val_loss = running_loss / len(dataloader)
    val_acc = 100. * correct / total
   
    return val_loss, val_acc

# ============================
# TRAINING
# ============================
print("\n" + "="*60)
print("OPTIMIZED AST TRAINING - Target: <3 hours, >0.80 F1")
print("="*60)

os.environ["WANDB_API_KEY"] = "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"

wandb.init(
    project="DL-GenAi-t1-2026",
    name="ast-optimized-fast",
    config={
        "model": "AST",
        "pretrained": True,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "samples_per_genre": SAMPLES_PER_GENRE_TRAIN,
        "num_workers": NUM_WORKERS,
        "optimizations": "no_tempo_sync, reduced_samples, larger_batch"
    }
)

# Get noise paths
noise_paths = get_noise_paths(DATA_PATH)
print(f"Found {len(noise_paths)} noise files")

# Create datasets
print("\nCreating datasets...")
train_base = OptimizedMashupDataset(
    DATA_PATH, GENRES, STEMS, noise_paths,
    num_samples_per_genre=SAMPLES_PER_GENRE_TRAIN
)

val_base = OptimizedMashupDataset(
    DATA_PATH, GENRES, STEMS, noise_paths,
    num_samples_per_genre=SAMPLES_PER_GENRE_VAL
)

# Load AST model and feature extractor
print("Loading pretrained AST model...")
feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

ast_model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=10,
    ignore_mismatched_sizes=True
).to(DEVICE)

# Wrap with AST dataset
train_ast_dataset = ASTDataset(train_base, feature_extractor)
val_ast_dataset = ASTDataset(val_base, feature_extractor)

train_loader = DataLoader(
    train_ast_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ast_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Train samples: {len(train_ast_dataset)}, Val samples: {len(val_ast_dataset)}")
print(f"Steps per epoch: {len(train_loader)}")

# Optimizer and Scheduler
optimizer = torch.optim.AdamW(ast_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1
)

# Training loop
print("\nStarting training...")
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")
   
    train_loss, train_acc = train_ast_epoch(ast_model, train_loader, optimizer, scheduler, DEVICE)
    val_loss, val_acc = validate_ast(ast_model, val_loader, DEVICE)
   
    print(f"\nTrain Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
   
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
   
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ast_model.save_pretrained("ast_best")
        print(f"✓ Saved best AST model (val_acc: {val_acc:.2f}%)")

wandb.finish()

print(f"\n{'='*60}")
print(f"✓ AST Training Complete!")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"{'='*60}")


OPTIMIZED AST TRAINING - Target: <3 hours, >0.80 F1


wandb: Currently logged in as: 24f1002246 (24f1002246-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 2lplg4qw
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260328_183926-2lplg4qw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run ast-optimized-fast
wandb: ⭐️ View project at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: 🚀 View run at https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/2lplg4qw


Found 2000 noise files

Creating datasets...
Loading pretrained AST model...


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Training AST:   0%|          | 0/94 [00:00<?, ?it/s]

Validating AST:   0%|          | 0/19 [00:00<?, ?it/s]


Train Loss: 0.0527, Train Acc: 98.33%
Val Loss: 0.0523, Val Acc: 98.00%


wandb: uploading history steps 9-9, summary, console lines 88-90
wandb: uploading data
wandb: 
wandb: Run history:
wandb:         epoch ▁▂▃▃▄▅▆▆▇█
wandb: learning_rate ██▇▆▅▄▃▂▁▁
wandb:     train_acc ▁▇▇▇██████
wandb:    train_loss █▃▂▂▂▁▁▁▁▁
wandb:       val_acc ▁▄▄▆▆▇▇██▇
wandb:      val_loss █▄▄▃▂▂▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 10
wandb: learning_rate 0.0
wandb:     train_acc 98.33333
wandb:    train_loss 0.05269
wandb:       val_acc 98
wandb:      val_loss 0.05227
wandb: 
wandb: 🚀 View run ast-optimized-fast at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026/runs/2lplg4qw
wandb: ⭐️ View project at: https://wandb.ai/24f1002246-indian-institute-of-technology-madras/DL-GenAi-t1-2026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260328_183926-2lplg4qw/logs



✓ AST Training Complete!
Best Validation Accuracy: 99.33%


In [8]:
print("\n" + "="*60)
print("STARTING INFERENCE WITH TTA")
print("="*60)

def get_multiple_crops(audio, duration_sec, sr=SAMPLE_RATE, num_crops=5):
    """Get multiple crops from audio for TTA"""
    target_length = duration_sec * sr
   
    if len(audio) < target_length:
        return [np.pad(audio, (0, target_length - len(audio)))]
   
    crops = []
    max_start = len(audio) - target_length
   
    if num_crops == 1:
        crops.append(audio[:target_length])
    else:
        step = max_start // (num_crops - 1) if num_crops > 1 else 0
       
        for i in range(num_crops):
            start = min(i * step, max_start)
            crop = audio[start:start + target_length]
            crops.append(crop)
   
    return crops
    
def predict_with_tta(audio, model, feature_extractor, num_crops=5):
    """Predict with Test-Time Augmentation"""
    crops = get_multiple_crops(audio, CROP_DURATION, SAMPLE_RATE, num_crops=num_crops)
   
    all_probs = []
   
    for crop in crops:
        inputs = feature_extractor(
            crop,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        )
       
        inputs = inputs.input_values.to(DEVICE)
       
        with torch.no_grad():
            outputs = model(inputs)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy()[0])
   
    # Average predictions
    avg_probs = np.mean(all_probs, axis=0)
    predicted_idx = np.argmax(avg_probs)
    predicted_genre = IDX_TO_GENRE[predicted_idx]
   
    return predicted_genre, avg_probs


STARTING INFERENCE WITH TTA


In [9]:
print("Loading best model for inference...")
ast_model = ASTForAudioClassification.from_pretrained("ast_best").to(DEVICE)
ast_model.eval()


test_df = pd.read_csv(DATA_PATH + "/test.csv")

print(f"Found {len(test_df)} test files")
print("Using TTA with 5 crops per audio")

results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Making predictions"):
   
    file_id = row["id"]
    file_path = DATA_PATH + "/" + row["filename"]
   
    audio = load_audio(file_path, SAMPLE_RATE)
   
    if audio is None:
        predicted_genre = "pop"  # Fallback
    else:
        predicted_genre, probs = predict_with_tta(
            audio, ast_model, feature_extractor, num_crops=5
        )
   
    results.append({
        "id": file_id,
        "genre": predicted_genre
    })

# ============================
# CREATE SUBMISSION
# ============================
print("\n" + "="*60)
print("CREATING SUBMISSION FILE")
print("="*60)

submission_df = pd.DataFrame(results)
submission_df = submission_df.sort_values("id").reset_index(drop=True)
submission_df.to_csv("submission.csv", index=False)

print(f"\n✓ Submission file created: submission.csv")
print(f"Total predictions: {len(submission_df)}")

print("\nFirst 10 predictions:")
print(submission_df.head(10))

print("\nGenre distribution:")
print(submission_df["genre"].value_counts())

print("\n" + "="*60)
print("✓ ALL COMPLETE!")
print("="*60)
print("Ready to submit to Kaggle!")

Loading best model for inference...
Found 3020 test files
Using TTA with 5 crops per audio


Making predictions:   0%|          | 0/3020 [00:00<?, ?it/s]


CREATING SUBMISSION FILE

✓ Submission file created: submission.csv
Total predictions: 3020

First 10 predictions:
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7       rock
7   8        pop
8   9        pop
9  10      disco

Genre distribution:
genre
hiphop       346
rock         344
disco        323
pop          313
jazz         313
metal        312
blues        309
reggae       300
country      249
classical    211
Name: count, dtype: int64

✓ ALL COMPLETE!
Ready to submit to Kaggle!
